<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/Topics/Gradient_Projected_Descent_Attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    "./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    "./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))
print("Image shape:", train_dataset[0][0].shape)

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.2MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 460kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.6MB/s]

Train: 60000
Test: 10000
Image shape: torch.Size([1, 28, 28])


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



class MNISTModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),

            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),

            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.network(x)


model = MNISTModel().to(device)

print(model)



criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{epochs}] "
        f"Loss: {total_loss / len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )


model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total

print(f"\nTest accuracy: {test_accuracy:.2f}%")


torch.save(model.state_dict(), "mnist_model.pth")

print("Model saved to mnist_model.pth")

Device: cuda
MNISTModel(
  (network): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=3136, out_features=128, bias=True)
    (8): ReLU()
    (9): Linear(in_features=128, out_features=10, bias=True)
  )
)
Epoch [1/5] Loss: 0.3626 Accuracy: 89.50%
Epoch [2/5] Loss: 0.0779 Accuracy: 97.65%
Epoch [3/5] Loss: 0.0513 Accuracy: 98.51%
Epoch [4/5] Loss: 0.0401 Accuracy: 98.75%
Epoch [5/5] Loss: 0.0324 Accuracy: 99.01%

Test accuracy: 98.97%
Model saved to mnist_model.pth


In [27]:
import torch
import torch.nn.functional as F


def gradient_projected_descent(
    model,
    images,
    epsilon=16/255,
    alpha=2/255,
    steps=40
):

    x_original = images.detach().clone()

    with torch.no_grad():
        target_labels = model(x_original).argmax(dim=1)

    x_adv = x_original + torch.empty_like(x_original).uniform_(
        -epsilon, epsilon
    )
    x_adv = torch.clamp(x_adv, 0, 1)

    for _ in range(steps):

        x_adv.requires_grad_(True)

        logits = model(x_adv)

        loss = F.cross_entropy(logits, target_labels)

        grad = torch.autograd.grad(loss, x_adv)[0]

        x_adv = x_adv.detach() + alpha * grad.sign()

        delta = torch.clamp(
            x_adv - x_original,
            -epsilon,
            epsilon
        )

        x_adv = torch.clamp(
            x_original + delta,
            0,
            1
        )

    with torch.no_grad():
        final_predictions = model(x_adv).argmax(dim=1)

    attack_success = final_predictions != target_labels

    return x_adv, attack_success

In [29]:
from tqdm import tqdm
from torch.utils.data import DataLoader

loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=False
)

results = []

for images, labels in tqdm(loader):

    images = images.to(device)

    adversarial_images, attack_success = gradient_projected_descent(
        model,
        images
    )

    results.extend(attack_success.cpu().tolist())

100%|██████████| 938/938 [01:04<00:00, 14.56it/s]


In [31]:
len(results)

60000

In [32]:
sum(results)/len(results)

0.0738